## Weakly Supervised Semantic Segmentation

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import torchvision.transforms.functional as tF

import copy

In [ ]:
USE_GPU = True
device = torch.device('cuda' if USE_GPU and torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Loading test data.

We chose the PASCAL VOC 2012 dataset. From our research, it appears to a popular reliable dataset that is not too large or complicated. It also has the bounding boxes required for WSSS.

In [ ]:
# SET TO FALSE WHEN DONE!
dataset = torchvision.datasets.VOCDetection(root='./data', year='2012', image_set='train', download=False)
NUM_CLASSES = 21
VOC_CLASSES = [
    "background",
    "aeroplane",
    "bicycle",
    "bird",
    "boat",
    "bottle",
    "bus",
    "car",
    "cat",
    "chair",
    "cow",
    "diningtable",
    "dog",
    "horse",
    "motorbike",
    "person",
    "pottedplant",
    "sheep",
    "sofa",
    "train",
    "tvmonitor"
]

In [ ]:
# function for extracting labels from image data
def extract_label_mask(data):
    class_to_idx = {cls: i for i, cls in enumerate(VOC_CLASSES)}
    mask = torch.zeros(len(VOC_CLASSES), dtype=bool)

    objects = data.get('annotation', {}).get('object', [])

    # Handle case where there's only one object (not a list)
    if isinstance(objects, dict):
        objects = [objects]

    for obj in objects:
        cls_name = obj.get('name')
        # if cls_name in class_to_idx:
        #     mask[class_to_idx[cls_name]] = True

        if isinstance(cls_name, list):
            for c in cls_name:
                if c in class_to_idx:
                    mask[class_to_idx[c]] = True
            continue

        if isinstance(cls_name, str):
            if cls_name in class_to_idx:
                mask[class_to_idx[cls_name]] = True
                
    mask[0] = 1 # background is always present
    return mask
    


In [ ]:
# function for displaying bounding boxes

def show_bounding_box(figure, target):
  for obj in target['annotation']['object']:
    x, y, x2, y2 = [int(i) for i in obj['bndbox'].values()]
    w = x2 - x
    h = y2 - y

    rect = patches.Rectangle(
      (x, y), w, h,
      linewidth=2,
      edgecolor='red',
      facecolor='none'
    )

    figure.add_patch(rect)

    # Add label
    figure.text(
        x, y - 0.02,          # slightly above the box
        obj['name'],
        color='red',
        fontsize=12,
        verticalalignment='bottom',
        bbox=dict(facecolor='white', alpha=0.5, edgecolor='none', pad=1)
    )


Creating sanity dataset

In [ ]:
# imageset for overfit_net

sanity_data = []
sanity_objects = {'person', 'chair', 'dog'}
for data in dataset:
  objects = data[1]['annotation']['object']
  obj_set = set()
  for obj in objects:
    obj_set.add(obj['name'])
  if sanity_objects.issuperset(obj_set) and len(obj_set) > 1:
    sanity_data.append(data)
  if len(sanity_data) >= 16:
    break

fig = plt.figure(figsize=(14,10))
for i in range(1, 6):
  img, target = sanity_data[i-1]

  ax = fig.add_subplot(1,5,i)
  plt.title(f'sanity imageset number {i}')
  ax.imshow(img)
  show_bounding_box(ax, target)


Data augmentation

In [ ]:
import torchvision
from torchvision.transforms import v2

# Transform specifications taken from DinoV3 documentation
# def make_transform(resize_size: int = 256):
#     to_tensor = v2.ToImage()
#     resize = v2.Resize((resize_size, resize_size), antialias=True)
#     to_float = v2.ToDtype(torch.float32, scale=True)
#     normalize = v2.Normalize(
#         mean=(0.485, 0.456, 0.406),
#         std=(0.229, 0.224, 0.225),
#     )
#     return v2.Compose([to_tensor, resize, to_float, normalize])

sanity_transforms = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

preprocess_transforms = transforms.Compose([
    transforms.RandomResizedCrop(256, (0.5, 1.5)),
    transforms.RandomHorizontalFlip(),
    transforms.Resize((256, 256), antialias=True),
    transforms.ToTensor()
])

train_transforms = transforms.Compose([
    # transforms.RandomResizedCrop(256, (0.5, 1.5)),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [ ]:
from torch.utils.data import Dataset, DataLoader, Subset

class myDataset(Dataset):
  def __init__(self, dataset, transform=sanity_transforms):
    self.data = dataset
    self.transform = transform

  def __len__(self):
    return len(self.data)

  def __getitem__(self, idx):
    img, target = self.data[idx]

    if self.transform:
        img = preprocess_transforms(img)
        altered_img = self.transform(img)

    labels = extract_label_mask(target)

    return altered_img, labels, img

TRAIN_BATCH_SIZE = 8

sanity_dataset = myDataset(sanity_data, train_transforms)
test_dataset = myDataset(Subset(dataset, range(200)), train_transforms)
train_dataset = myDataset(dataset, train_transforms)

sanity_loader = DataLoader(sanity_dataset, batch_size=4, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=TRAIN_BATCH_SIZE, shuffle=True)
train_loader = DataLoader(train_dataset, batch_size=TRAIN_BATCH_SIZE, shuffle=True)

In [ ]:
# --- Image transform (match training normalization) ---
img_transform = transforms.Compose([
    transforms.Resize((256, 256)),   # match your training pipeline
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

# --- Mask transform (VERY IMPORTANT: NEAREST) ---
def target_transform(mask):
    mask = transforms.Resize(
        (256, 256),
        interpolation=transforms.InterpolationMode.NEAREST
    )(mask)
    return torch.as_tensor(np.array(mask), dtype=torch.long)

# --- Validation dataset (segmentation, not detection) ---
# SET TO FALSE WHEN DONE!
val_dataset = torchvision.datasets.VOCSegmentation(
    root='./data',year='2012',
    image_set='val',
    download=False,
    transform=img_transform,
    target_transform=target_transform
)

# --- Validation loader ---
val_loader = DataLoader(
    val_dataset,
    batch_size=4,        # adjust based on GPU
    shuffle=False,       # IMPORTANT: no shuffling for validation
)

In [ ]:
# model = torch.hub.load("facebookresearch/dinov3", "dinov3_vitb16")
vits_16_url = "https://dinov3.llamameta.net/dinov3_vits16/dinov3_vits16_pretrain_lvd1689m-08c60483.pth?Policy=eyJTdGF0ZW1lbnQiOlt7InVuaXF1ZV9oYXNoIjoiZW42ZXU2OGp5ZGowcWp4dTgzejQ2bzdoIiwiUmVzb3VyY2UiOiJodHRwczpcL1wvZGlub3YzLmxsYW1hbWV0YS5uZXRcLyoiLCJDb25kaXRpb24iOnsiRGF0ZUxlc3NUaGFuIjp7IkFXUzpFcG9jaFRpbWUiOjE3NzY0ODUxMzF9fX1dfQ__&Signature=lInKrWlw9YimwX87yq4EvvOlIXY5ClwzqJVSQe6ouURgePVWAuJgqMiAKt1hhHYSiUr8o%7ExxoUCvlM1jTPRyz9HIuZDDowWY7B2aOazw8UQWGa0jdni-hwwKf77BjsNOh8LfePhYlT0wkC461iPxSoWxp1AkLijH7vdLfsMO-JX-tVqAxQmZe3PULvG%7EErfgwoCn3ylL4wwGfBDzb8vB1OR81aCJvQNqrmISfuQrxkrqq9u0yfDJotlM4d0qIbGiw9JzZ57FtkqWo9KOobR%7EpniZzvPdl3xKwxaFYzo9j01CQWLT4YUU8pC3P90Ch1W7WXNShPiVrvcLluEToCYSUA__&Key-Pair-Id=K15QRJLYKIFSLZ&Download-Request-ID=1695652418115315"

dino_backbone = torch.hub.load(
    "facebookresearch/dinov3",
    "dinov3_vits16",
    source="github",
    weights=vits_16_url
)

print(dino_backbone)

dino_backbone.to(device)


In [ ]:
class DinoNuggets(nn.Module):
  def __init__(self, backbone, num_classes, freeze=True, embed_dim=384):
    super(DinoNuggets, self).__init__()
    self.backbone = backbone
    self.backbone_patch_size = 16

    # Freeze backbone weights
    if freeze:
      for name, param in self.backbone.named_parameters():
        param.requires_grad = False
    else:
      for name, param in self.backbone.named_parameters():
        param.requires_grad = True

    self.conv_1 = nn.Sequential(
      nn.Conv2d(embed_dim, 256, kernel_size=3, padding=1, bias=False),
      nn.ReLU(inplace=True),
    )

    self.conv_2 = nn.Sequential(
      nn.Conv2d(256, 256, kernel_size=3, padding=1, bias=False),
      nn.ReLU(inplace=True),
    )

    self.conv_3 = nn.Sequential(
      nn.Conv2d(256, 256, kernel_size=3, padding=1, bias=False),
      nn.ReLU(inplace=True),
    )

    self.final = nn.Conv2d(256, num_classes, kernel_size=(1, 1))

  def forward(self, x):
    B, _, H, W = x.shape

    x0 = self.backbone.get_intermediate_layers(
        x, n=1, reshape=True, return_class_token=False, norm=True
    )[0]  # (B, D, h_patch, w_patch)

    _, _, h_patch, w_patch = x0.shape
    N = h_patch * w_patch

    patch_tokens, cls_token = self.backbone.get_intermediate_layers(
        x, n=1, reshape=False, return_class_token=True, norm=True
    )[0]

    # Dynamically skip however many register tokens are prepended
    num_registers = patch_tokens.shape[1] - N
    patch_tokens = patch_tokens[:, num_registers:, :]              # (B, N, D)

    cls_norm   = F.normalize(cls_token, dim=-1).unsqueeze(1)      # (B, 1, D)
    patch_norm = F.normalize(patch_tokens, dim=-1)                 # (B, N, D)
    attn = (cls_norm * patch_norm).sum(dim=-1)                    # (B, N)
    attn = attn.reshape(B, 1, h_patch, w_patch)
    attn = F.interpolate(attn, size=(H, W), mode="bilinear", align_corners=False)
    attn = (attn - attn.min()) / (attn.max() - attn.min() + 1e-5)

    x1 = self.conv_1(x0)
    # x2 = self.conv_2(x1)
    x3 = self.conv_3(x1)

    features = F.interpolate(x3, size=(H, W), mode="bilinear", align_corners=False)
    xfinal = self.final(features)

    if self.training:
        return xfinal, attn.detach()
    return xfinal

In [ ]:
from PIL import Image, ImageOps

# Define colour palette for visualization of segmentation
palette = [0, 0, 0, 128, 0, 0, 0, 128, 0, 128, 128, 0, 0, 0, 128, 128, 0, 128, 0, 128, 128,
           128, 128, 128, 64, 0, 0, 192, 0, 0, 64, 128, 0, 192, 128, 0, 64, 0, 128, 192, 0, 128,
           64, 128, 128, 192, 128, 128, 0, 64, 0, 128, 64, 0, 0, 192, 0, 128, 192, 0, 0, 64, 128]

def colorize_mask(mask):
    new_mask = Image.fromarray(mask.astype(np.uint8)).convert('P')
    new_mask.putpalette(palette)

    return new_mask

In [ ]:
def show_CAM(net, imageset, class_id, sample_id, normalize=True):
    # Extract CAM (logits → probabilities)
    net.eval()
    
    img, img_data = imageset[sample_id]
        
    output = net.forward(sanity_transforms(img).unsqueeze(0).to(device))
    print(output.shape)
    cam = torch.sigmoid(output[0, class_id])  # (H, W)

    if normalize:
        cam = cam - cam.min()
        cam = cam / (cam.max() + 1e-5)

    cam = cam.detach().cpu().numpy()

    plt.figure(figsize=(5, 5))
    plt.imshow(cam, cmap='jet')
    plt.colorbar()
    plt.title(f"CAM - class {class_id}")
    plt.axis('off')
    plt.show()

In [ ]:
def tag_consistency(result, target, r):
    fg_logits = result[:, 1:] # ignore background
    pooled = (1/r) * torch.logsumexp(r * fg_logits.flatten(2), dim=-1)  # (B, 20)
    return F.binary_cross_entropy_with_logits(pooled, target[:, 1:].float())


def tag_loss(result, target, bg_weight):
    """
    result: (B, C, H, W) logits
    target: (B, C) multi-hot tensor from DataLoader
    bg_weight: penalty weight for background

    returns: scalar loss
    """

    # ensure target is on same device
    class_mask = target.to(result.device).bool()  # (B, C)

    # log probabilities
    log_probs = F.softmax(result, dim=1)  # (B, C, H, W)

    # expand mask
    mask = class_mask[:, :, None, None]  # (B, C, 1, 1)

    # mask out absent classes
    masked_log_probs = log_probs.masked_fill(~mask, float('-inf'))

    # main loss: log(sum of present class probs)
    pixel_log_prob = torch.logsumexp(masked_log_probs, dim=1)  # (B, H, W)
    main_loss = -pixel_log_prob.mean()

    # --- background penalty ---
    bg_idx = 0  # VOC: background is index 0

    probs = torch.softmax(result, dim=1)
    bg_prob = probs[:, bg_idx, :, :]  # (B, H, W)

    bg_loss = bg_prob.mean()

    # total loss
    loss = main_loss + (bg_weight * bg_loss)**2

    return loss

def seed_loss(result, target, threshold=0.6):
    """
    Weak localization loss using CAMs (ignores background class 0).

    Args:
        result: (B, C, H, W) logits
        target: (B, C) image-level labels (0/1)
        threshold: confidence threshold for CAM

    Returns:
        scalar loss
    """

    B, C, H, W = result.shape

    # --- 1. Get CAMs (ignore background channel) ---
    # cams = torch.sigmoid(result[:, 1:])  # (B, C-1, H, W)
    cams = torch.sigmoid(result)  # (B, C, H, W)

    # --- 2. Mask out absent classes (ignore background label too) ---
    # cams = cams * target[:, 1:].unsqueeze(-1).unsqueeze(-1)
    cams = cams * target.unsqueeze(-1).unsqueeze(-1)

    # --- 3. Normalize per class ---
    # cams_flat = cams.view(B, C-1, -1)
    # cam_min = cams_flat.min(dim=-1)[0].view(B, C-1, 1, 1)
    # cam_max = cams_flat.max(dim=-1)[0].view(B, C-1, 1, 1)
    cams_flat = cams.view(B, C, -1)
    cam_min = cams_flat.min(dim=-1)[0].view(B, C, 1, 1)
    cam_max = cams_flat.max(dim=-1)[0].view(B, C, 1, 1)
    cams = (cams - cam_min) / (cam_max - cam_min + 1e-5)

    # --- 4. Get pseudo labels from foreground classes only ---
    max_vals, pseudo_labels = cams.max(dim=1)  # (B, H, W)

    # shift labels by +1 to match original class indices
    pseudo_labels = pseudo_labels# + 1  # now in [1, C-1]

    # --- 5. Select confident pixels ---
    mask = max_vals > threshold  # (B, H, W)

    if mask.sum() == 0:
        return torch.tensor(0.0, device=result.device, requires_grad=True)

    # --- 6. Compute log probabilities ---
    log_probs = F.log_softmax(result, dim=1)  # (B, C, H, W)

    # --- 7. Gather NLL for chosen class ---
    nll = -log_probs.gather(1, pseudo_labels.unsqueeze(1)).squeeze(1)

    # --- 8. Apply mask and average ---
    loss = nll[mask].mean()

    return loss

def global_max_pool_loss(result, target):
    processed = torch.softmax(result, dim=1)
    pooled = torch.amax(processed, dim=(2, 3))
    
    # --- 2. Multi-label classification loss ---
    loss = F.binary_cross_entropy(pooled, target.float())

    return loss

def global_avg_pool_loss(result, target):
    processed = torch.softmax(result, dim=1)
    pooled = torch.mean(processed, dim=(2, 3))
    
    # --- 2. Multi-label classification loss ---
    loss = F.binary_cross_entropy(pooled, target.float())

    return loss

def cam_loss(result, attn, target, low_threshold, high_threshold):
  B, _, H, W = result.shape

  fg_target = target[:, 1:].unsqueeze(-1).unsqueeze(-1) # (B, 20, 1, 1)

  fg_cams = torch.sigmoid(result[:, 1:]) # (B, 20, H, W)
  fg_cams = fg_cams * fg_target # zero absent classes

  flat = fg_cams.flatten(2) # (B, 20, H*W)
  cam_min = flat.min(dim=-1)[0].view(B, 20, 1, 1)
  cam_max = flat.max(dim=-1)[0].view(B, 20, 1, 1)
  fg_cams = (fg_cams - cam_min) / (cam_max - cam_min + 1e-5)

  # deal with background separately
  bg_score = 1.0 - attn # (B, 1, H, W)

  all_scores = torch.cat([bg_score, fg_cams], dim=1) # (B, 21, H, W)
  max_scores, pseudo_labels = all_scores.max(dim=1) # (B, H, W)

  pseudo_labels[max_scores < high_threshold] = 255
  pseudo_labels[attn.squeeze(1) < low_threshold] = 0 

  loss = F.cross_entropy(result, pseudo_labels, ignore_index=255)

  return loss

def crf_spatial_loss(result, image, sigma_xy=3.0, sigma_rgb=10.0, kernel_size=3):
    """
    CRF-inspired spatial regularization loss.

    Args:
        result: (B, C, H, W) logits
        image:  (B, 3, H, W) input image (normalized or [0,1])
        sigma_xy: spatial Gaussian std
        sigma_rgb: color Gaussian std
        kernel_size: neighborhood size (odd number)

    Returns:
        scalar loss
    """

    B, C, H, W = result.shape

    # --- 1. Convert logits to probabilities ---
    probs = F.softmax(result, dim=1)  # (B, C, H, W)

    # --- 2. Extract local neighborhoods ---
    pad = kernel_size // 2

    probs_unfold = F.unfold(probs, kernel_size, padding=pad)  # (B, C*K*K, H*W)
    image_unfold = F.unfold(image, kernel_size, padding=pad)  # (B, 3*K*K, H*W)

    # reshape
    probs_unfold = probs_unfold.view(B, C, kernel_size*kernel_size, H, W)
    image_unfold = image_unfold.view(B, 3, kernel_size*kernel_size, H, W)

    # center pixel
    probs_center = probs.unsqueeze(2)   # (B, C, 1, H, W)
    image_center = image.unsqueeze(2)   # (B, 3, 1, H, W)

    # --- 3. Compute pairwise differences ---
    prob_diff = (probs_center - probs_unfold).pow(2).sum(dim=1)  # (B, K*K, H, W)
    color_diff = (image_center - image_unfold).pow(2).sum(dim=1)  # (B, K*K, H, W)

    # --- 4. Spatial Gaussian kernel ---
    coords = torch.stack(torch.meshgrid(
        torch.arange(kernel_size),
        torch.arange(kernel_size),
        indexing="ij"
    ), dim=-1).float().to(result.device)

    coords = coords - kernel_size // 2
    spatial_dist = (coords[..., 0]**2 + coords[..., 1]**2).view(1, -1, 1, 1)

    spatial_kernel = torch.exp(-spatial_dist / (2 * sigma_xy**2))  # (1, K*K, 1, 1)

    # --- 5. Bilateral kernel (spatial + color) ---
    bilateral_kernel = spatial_kernel * torch.exp(-color_diff / (2 * sigma_rgb**2))

    # --- 6. Final loss ---
    loss = (bilateral_kernel * prob_diff).mean()

    return loss

def bg_loss(result, weight = 0.1):
    bg_idx = 0  # VOC: background is index 0

    probs = torch.softmax(result, dim=1)
    bg_prob = probs[:, bg_idx, :, :]  # (B, H, W)

    return weight * bg_prob.mean()

TAG_COEF = 2.0
CAM_COEF = 1.0
SPACE_COEF = 1.0

def loss(result, attn, target, image):
    loss = 0
    # loss += TAG_COEF * tag_consistency(result, target, 10) 
    # loss += CAM_COEF * cam_loss(result, attn, target, 0.1, 0.7)
    loss += 0.1 * tag_loss(result, target, 1)
    loss += CAM_COEF * seed_loss(result, target, threshold=0.7)
    loss += TAG_COEF * global_avg_pool_loss(result, target)
    loss += TAG_COEF * global_max_pool_loss(result, target)
    loss += SPACE_COEF * crf_spatial_loss(result, image)
    # loss += bg_loss(result, weight=0.1)
    return loss


In [ ]:
def get_optimizer(net):
    optimizer = torch.optim.Adam(net.parameters(), lr=0.0001)
    return optimizer

In [ ]:
def train(train_loader, net, optimizer, loss_graph, criterion = loss):

    net.train()

    total_loss = 0

    for i, data in enumerate(train_loader):

        inputs, target, img = data
        inputs = inputs.to(device)
        target = target.to(device)
        img = img.to(device)

        optimizer.zero_grad()

        outputs, attn = net.forward(inputs)
        loss = criterion(outputs, attn, target, img)
        loss.backward()
        optimizer.step()

        total_loss += loss

        loss_graph.append(loss.item())

    # Return the value of loss (averaged over the loaded data)
    val_loss = total_loss / len(train_loader)
    return val_loss

In [ ]:
def run_training(net, num_epochs, loader):
    # switch to train mode (original untrained_net was set to eval mode)
    net.train()

    optimizer = get_optimizer(net)

    print("Starting Training...")

    loss_graph = []

    fig = plt.figure(figsize=(12,6))
    plt.subplots_adjust(bottom=0.2,right=0.85,top=0.95)
    ax = fig.add_subplot(1,1,1)

    for e in range(num_epochs):
        loss = train(loader, net, optimizer, loss_graph)
        ax.clear()
        ax.set_xlabel('iterations')
        ax.set_ylabel('loss value')
        ax.set_title('Training loss curve for net')
        ax.plot(loss_graph, label='training loss')
        ax.legend(loc='upper right')
        fig.canvas.draw()
        print("Epoch: {} Loss: {}".format(e, loss))

In [ ]:
def show_training_results(net, index, imageset):
    net.eval()
    
    img, img_data = imageset[index]
        
    output = net.forward(sanity_transforms(img).unsqueeze(0).to(device))

    fig = plt.figure(figsize=(14, 10))
    ax = fig.add_subplot(1, 2, 1)
    plt.title('image sample')
    ax.imshow(img)
    show_bounding_box(ax, img_data)

    ax = fig.add_subplot(1, 2, 2)
    plt.title('network output/prediction')

    # Display the segmentation mask with colorized labels
    segmentation_mask = torch.argmax(output, dim=1).cpu().numpy()[0]
    ax.imshow(colorize_mask(segmentation_mask))

    # Add a legend for the classes actually present in the image
    from matplotlib.lines import Line2D
    unique_labels = np.unique(segmentation_mask)
    legend_elements = [
        Line2D([0], [0], color=np.array(palette[label * 3:label * 3 + 3]) / 255, lw=4, label=VOC_CLASSES[label])
        for label in unique_labels if label < len(VOC_CLASSES)
    ]
    ax.legend(handles=legend_elements, loc='upper right', bbox_to_anchor=(1.5, 1))

In [ ]:
untrained_net = DinoNuggets(dino_backbone, NUM_CLASSES).to(device).eval()

overfit_net = copy.deepcopy(untrained_net).to(device)

# You can change the number of EPOCHS
run_training(overfit_net, num_epochs=20, loader=sanity_loader)

In [ ]:
 # Test untrained model on sample image
sample_index = 4
show_training_results(untrained_net, index=sample_index, imageset=sanity_data)

In [ ]:
# Show attention map as test
show_CAM(overfit_net, sanity_data, 15, sample_index)

In [ ]:
show_training_results(overfit_net, index=sample_index, imageset=sanity_data)

In [ ]:
test_net = DinoNuggets(dino_backbone, NUM_CLASSES).to(device)
run_training(test_net, num_epochs=10, loader=test_loader)

In [ ]:
# Test test model on sample image
show_training_results(test_net, index=7, imageset=dataset)

In [ ]:
big_net = copy.deepcopy(untrained_net).to(device)
# run_training(big_net, num_epochs=10, loader=train_loader)

In [ ]:
# Test overall model on sample image
show_training_results(big_net, index=125, imageset=dataset)

In [ ]:
from torchmetrics.segmentation import MeanIoU

miou = MeanIoU(num_classes=21).to(device)

validation_criterion = nn.CrossEntropyLoss(ignore_index=255)

def validate(val_loader, net, criterion=validation_criterion):
    net.eval()
    miou.reset()

    total_loss = 0.0
    num_batches = 0

    with torch.no_grad():
        for inputs, gts in val_loader:
            inputs = inputs.to(device)
            gts = gts.to(device)

            outputs = net(inputs)
            loss = criterion(outputs, gts)
            preds = torch.argmax(outputs, dim=1)

            valid_mask = (gts != 255)

            preds = preds.clone()
            gts = gts.clone()

            preds[~valid_mask] = 0
            gts[~valid_mask] = 0
            miou.update(preds, gts)

            total_loss += loss.item()
            num_batches += 1

    val_loss = total_loss / num_batches
    val_miou = miou.compute()

    return val_loss, val_miou

In [ ]:
val_loss, val_miou = validate(val_loader, test_net)
print("Validation loss:", val_loss)
print("Validation MIoU:", val_miou)